In [ ]:
import random, sys, time

###########################################################################
#                                                                         #
# Implement a hash table from scratch! (⑅•ᴗ•⑅)                            #
#                                                                         #
# Please do not use Python's dictionary or Python's collections library.  #
# The goal is to implement the data structure yourself.                   #
#                                                                         #
###########################################################################

# Hash function.
#
# |key|: string
# Return value: a hash value
def calculate_hash(key):
    assert type(key) == str
    # Note: This is not a good hash function. Do you see why?
    hash = 0
    for i in key:
        hash += ord(i)
    return hash

def generate_prime(num):
    if num == 1:
        return 1
    prime_list = [2]
    for i in range(num + 1):
        for prime in prime_list:
            if i % prime == 0:
                break
        else:
            prime_list.append(i)
    return prime_list[-1] 

def expand_or_shrink_hash(self, is_expand):
    if is_expand:
        new_bucket_size = generate_prime(self.bucket_size*2)
    else:
        new_bucket_size = generate_prime(self.bucket_size//2)
    new_hash_table = HashTable(bucket_size = new_bucket_size)
    
    for bucket in new_hash_table.buckets:
        while bucket:
            new_hash_table.put(bucket.key, bucket.value)
            bucket = bucket.next
    
    self.bucket_size = new_hash_table.bucket_size
    self.buckets = new_hash_table.buckets
    self.item_count = new_hash_table.item_count

# An item object that represents one key - value pair in the hash table.
class Item:
    # |key|: The key of the item. The key must be a string.
    # |value|: The value of the item.
    # |next|: The next item in the linked list. If this is the last item in the
    #         linked list, |next| is None.
    def __init__(self, key, value, next):
        assert type(key) == str
        self.key = key
        self.value = value
        self.next = next


# The main data structure of the hash table that stores key - value pairs.
# The key must be a string. The value can be any type.
#
# |self.bucket_size|: The bucket size.
# |self.buckets|: An array of the buckets. self.buckets[hash % self.bucket_size]
#                 stores a linked list of items whose hash value is |hash|.
# |self.item_count|: The total number of items in the hash table.
class HashTable:

    # Initialize the hash table.
    def __init__(self, bucket_size = 97):
        # Set the initial bucket size to 97. A prime number is chosen to reduce
        # hash conflicts.
        self.bucket_size = bucket_size
        self.buckets = [None] * self.bucket_size
        self.item_count = 0

    # Put an item to the hash table. If the key already exists, the
    # corresponding value is updated to a new value.
    #
    # |key|: The key of the item.
    # |value|: The value of the item.
    # Return value: True if a new item is added. False if the key already exists
    #               and the value is updated.
    def put(self, key, value):
        assert type(key) == str
        self.check_size() # Note: Don't remove this code.
        bucket_index = calculate_hash(key) % self.bucket_size
        item = self.buckets[bucket_index]
        while item:
            if item.key == key:
                item.value = value
                return False
            item = item.next
        new_item = Item(key, value, self.buckets[bucket_index])
        tmp = self.buckets[bucket_index]
        self.buckets[bucket_index] = new_item
        new_item.next = tmp
        self.item_count += 1
        if self.item_count // self.bucket_size > 0.3:
            expand_or_shrink_hash(self, True)
        return True

    # Get an item from the hash table.
    #
    # |key|: The key.
    # Return value: If the item is found, (the value of the item, True) is
    #               returned. Otherwise, (None, False) is returned.
    def get(self, key):
        assert type(key) == str
        self.check_size() # Note: Don't remove this code.
        bucket_index = calculate_hash(key) % self.bucket_size
        item = self.buckets[bucket_index]
        while item:
            if item.key == key:
                return (item.value, True)
            item = item.next
        return (None, False)

    # Delete an item from the hash table.
    #
    # |key|: The key.
    # Return value: True if the item is found and deleted successfully. False
    #               otherwise.
    def delete(self, key):
        assert type(key) == str
        self.check_size() # Note: Don't remove this code.
        bucket_index = calculate_hash(key) % self.bucket_size
        item = self.buckets[bucket_index]
        while item:
            if item.key == key:
                if item.next:
                    item.value = item.next
                else:
                    item.key = None
                    item.value = None
                self.item_count -= 1
                if self.item_count // self.bucket_size < 0.3:
                    expand_or_shrink_hash(self, False)
                return True
            
            item = item.next
            
        return False

    # Return the total number of items in the hash table.
    def size(self):
        return self.item_count

    # Check that the hash table has a "reasonable" bucket size.
    # The bucket size is judged "reasonable" if it is smaller than 100 or
    # the buckets are 30% or more used.
    #
    # Note: Don't change this function.
    def check_size(self):
        assert (self.bucket_size < 100 or
                self.item_count >= self.bucket_size * 0.3)


# Test the functional behavior of the hash table.
def functional_test():
    hash_table = HashTable()

    assert hash_table.put("aaa", 1) == True
    assert hash_table.get("aaa") == (1, True)
    assert hash_table.size() == 1

    assert hash_table.put("bbb", 2) == True
    assert hash_table.put("ccc", 3) == True
    assert hash_table.put("ddd", 4) == True
    assert hash_table.get("aaa") == (1, True)
    assert hash_table.get("bbb") == (2, True)
    assert hash_table.get("ccc") == (3, True)
    assert hash_table.get("ddd") == (4, True)
    assert hash_table.get("a") == (None, False)
    assert hash_table.get("aa") == (None, False)
    assert hash_table.get("aaaa") == (None, False)
    assert hash_table.size() == 4

    assert hash_table.put("aaa", 11) == False
    assert hash_table.get("aaa") == (11, True)
    assert hash_table.size() == 4

    assert hash_table.delete("aaa") == True
    assert hash_table.get("aaa") == (None, False)
    assert hash_table.size() == 3

    assert hash_table.delete("a") == False
    assert hash_table.delete("aa") == False
    assert hash_table.delete("aaa") == False
    assert hash_table.delete("aaaa") == False

    assert hash_table.delete("ddd") == True
    assert hash_table.delete("ccc") == True
    assert hash_table.delete("bbb") == True
    assert hash_table.get("aaa") == (None, False)
    assert hash_table.get("bbb") == (None, False)
    assert hash_table.get("ccc") == (None, False)
    assert hash_table.get("ddd") == (None, False)
    assert hash_table.size() == 0

    assert hash_table.put("abc", 1) == True
    assert hash_table.put("acb", 2) == True
    assert hash_table.put("bac", 3) == True
    assert hash_table.put("bca", 4) == True
    assert hash_table.put("cab", 5) == True
    assert hash_table.put("cba", 6) == True
    assert hash_table.get("abc") == (1, True)
    assert hash_table.get("acb") == (2, True)
    assert hash_table.get("bac") == (3, True)
    assert hash_table.get("bca") == (4, True)
    assert hash_table.get("cab") == (5, True)
    assert hash_table.get("cba") == (6, True)
    assert hash_table.size() == 6

    assert hash_table.delete("abc") == True
    assert hash_table.delete("cba") == True
    assert hash_table.delete("bac") == True
    assert hash_table.delete("bca") == True
    assert hash_table.delete("acb") == True
    assert hash_table.delete("cab") == True
    assert hash_table.size() == 0
    print("Functional tests passed!")


# Test the performance of the hash table.
#
# Your goal is to make the hash table work with mostly O(1).
# If the hash table works with mostly O(1), the execution time of each iteration
# should not depend on the number of items in the hash table. To achieve the
# goal, you will need to 1) implement rehashing (Hint: expand / shrink the hash
# table when the number of items in the hash table hits some threshold) and
# 2) tweak the hash function (Hint: think about ways to reduce hash conflicts).
def performance_test():
    hash_table = HashTable()

    for iteration in range(100):
        begin = time.time()
        random.seed(iteration)
        for i in range(10000):
            rand = random.randint(0, 100000000)
            hash_table.put(str(rand), str(rand))
        random.seed(iteration)
        for i in range(10000):
            rand = random.randint(0, 100000000)
            hash_table.get(str(rand))
        end = time.time()
        print("%d %.6f" % (iteration, end - begin))

    for iteration in range(100):
        random.seed(iteration)
        for i in range(10000):
            rand = random.randint(0, 100000000)
            hash_table.delete(str(rand))

    assert hash_table.size() == 0
    print("Performance tests passed!")



hash_table = HashTable()
    
hash_table.put("elica", 1)
print(hash_table.get("elica"))
hash_table.put("alice", 2)
print(hash_table.get("elica"))
print(hash_table.get("alice"))
    


(1, True)
(1, True)
(2, True)


In [15]:
import math

prime_list = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]

def find_near_prime(num):
    for p in prime_list:
        if p >= num:
            return p

    for integer in range(prime_list[-1], int(num*2), 2):
        is_prime = True
        for prime in prime_list:
            if prime >= math.sqrt(integer):
                break
            elif integer % prime == 0:
                is_prime = False
                break
        if is_prime:
            # print(integer)
            prime_list.append(integer)
            if integer >= num:
                return integer
    return prime_list[-1]

def main():
    near_prime = find_near_prime(20)
    print(near_prime)

main()

23


In [8]:
while True:
    line = input()
    answer = 0
    index = 0
    is_plus = True
    while index < len(line):
        number = 0
        if line[index].isdigit():
            while index < len(line) and line[index].isdigit():
                number = number * 10 + int(line[index])
                index += 1
            print(index)
            if line[index] == '.':
                while index < len(line) and line[index].isdigit():
                    decimal_place = -1
                    number = int(line[index]) * (10 ** decimal_place) + number
                    decimal_place -= 1
                    index += 1
            if is_plus:
                answer += number 
            else:
                answer -= number 
        elif line[index] == '+':
            is_plus = True
            index += 1
        elif line[index] == '-':
            is_plus = False
            index += 1
        else:
            print('Invalid character found: ' + line[index])
            exit(1)
    print("answer = %d\n" % answer)


1
3


IndexError: string index out of range

In [4]:

line = input()
answer = 0.0 # 小数点を考慮して浮動小数点数に
index = 0
is_plus = True

while index < len(line):
    if line[index].isdigit():
        number = 0.0 
        while index < len(line) and line[index].isdigit():
            number = number * 10 + int(line[index])
            index += 1
            
        if index < len(line) and line[index] == '.':
            index += 1 
            decimal_place = 0.1
            while index < len(line) and line[index].isdigit():
                number += int(line[index]) * decimal_place
                decimal_place *= 0.1 # 次の桁のために10分の1にする
                index += 1
        if is_plus:
            answer += number 
        else:
            answer -= number 
    elif line[index] == '+':
        is_plus = True
        index += 1
    elif line[index] == '-':
        is_plus = False
        index += 1
    else:
        print('Invalid character found: ' + line[index])
        exit(1)
print("answer = %.2f\n" % answer)

answer = 2.02

